In [13]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV_PATH = "data/XAU_1m_data.csv"   # กำหนดตรงๆ เวลารันใน notebook
OUT_DIR = "eda_output"
os.makedirs(OUT_DIR, exist_ok=True)

In [14]:
# ---------- 1. Load ----------
df = pd.read_csv(CSV_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

In [8]:
# ---------- 2. Data quality checks ----------
print(f"Rows: {len(df):,}  |  Range: {df['Date'].min()} -> {df['Date'].max()}")
print("Missing values:\n", df.isna().sum())
print("Duplicate timestamps:", df["Date"].duplicated().sum())
 
# OHLC logical consistency: High must be max, Low must be min of the bar
bad_ohlc = df[
    (df["High"] < df["Low"])
    | (df["Open"] > df["High"]) | (df["Open"] < df["Low"])
    | (df["Close"] > df["High"]) | (df["Close"] < df["Low"])
]
print("Rows with broken OHLC logic:", len(bad_ohlc))
 
# Non-positive prices / zero volume
print("Non-positive prices:", (df[["Open", "High", "Low", "Close"]] <= 0).sum().sum())
print("Zero-volume bars:", (df["Volume"] == 0).sum())
 
# Time gaps (missing minutes) — flags weekends/holidays and data outages
gaps = df["Date"].diff()
big_gaps = gaps[gaps > pd.Timedelta(hours=6)]
print(f"Gaps > 6h: {len(big_gaps)}  |  Largest gap: {gaps.max()}")

Rows: 6,822,714  |  Range: 2004-06-11 07:18:00 -> 2026-02-27 05:41:00
Missing values:
 Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
Duplicate timestamps: 0
Rows with broken OHLC logic: 0
Non-positive prices: 0
Zero-volume bars: 0
Gaps > 6h: 1181  |  Largest gap: 32 days 08:12:00


In [9]:
# ---------- 3. Returns & volatility ----------
df["ret"] = df["Close"].pct_change()
print("\n1-min return stats:\n", df["ret"].describe())
extreme = df.loc[df["ret"].abs() > 0.05, ["Date", "Close", "ret"]]
print(f"\n|return| > 5% events: {len(extreme)}")
print(extreme)


1-min return stats:
 count    6.822713e+06
mean     4.374044e-07
std      3.341336e-04
min     -3.327759e-02
25%     -1.138878e-04
50%      0.000000e+00
75%      1.151676e-04
max      1.474595e-01
Name: ret, dtype: float64

|return| > 5% events: 1
                       Date    Close       ret
6695560 2025-10-15 07:59:00  4179.69  0.147459


In [10]:
# ---------- 4. Resample to daily for readable plots ----------
d = df.set_index("Date")
daily = d["Close"].resample("D").last().dropna()
daily_vol = d["Volume"].resample("D").sum()

In [11]:
# ---------- 5. Yearly summary table ----------
df["Year"] = df["Date"].dt.year
yearly = df.groupby("Year").agg(
    open=("Open", "first"), close=("Close", "last"),
    high=("High", "max"), low=("Low", "min"),
    avg_vol=("Volume", "mean"), bars=("Close", "count"),
)
print("\nYearly summary:\n", yearly)
yearly.to_csv(f"{OUT_DIR}/yearly_summary.csv")


Yearly summary:
          open     close      high      low     avg_vol    bars
Year                                                          
2004   384.00   437.000   456.300   381.10    2.836784   79588
2005   437.10   514.600   540.500   409.80    2.217233  140683
2006   516.80   636.200   729.700   516.30    2.748635  166248
2007   638.20   831.800   845.400   601.50    5.239017  241895
2008   841.00   866.800  1032.200   681.40    7.754170  315462
2009   874.20  1097.510  1226.370   801.90   23.692609  329281
2010  1094.61  1408.130  1430.880  1044.46   37.745121  316648
2011  1415.71  1564.110  1920.610  1307.88   41.754443  323126
2012  1567.37  1674.870  1795.870  1526.96   85.877195  329914
2013  1673.25  1208.260  1696.100  1180.17  123.504304  348635
2014  1225.12  1186.830  1388.910  1131.44   46.930785  348450
2015  1186.94  1060.810  1307.530  1046.23   61.485055  346707
2016  1064.88  1151.320  1375.050  1061.66   86.567591  350238
2017  1151.67  1302.330  1357.420  11